# Google Colab Setup

Run vehicle tracking and make/model census on Google Colab T4 GPU.

## 1. Get Car-Census code

In [ ]:
from google.colab import userdata
import os

# 1. Retrieve the secret securely
token = userdata.get('GITHUB_TOKEN')

# 2. Define the repo details
# Note: Remove 'https://' from the start of your repo string for the formatting below
repo_path = "github.com/DmitryMatv/Car-Census.git"
repo_url = f"https://{token}@{repo_path}"

# 3. Clone the repo
!git clone {repo_url}
    
# 4. Change directory to the cloned repo
os.chdir('Car-Census')

In [ ]:
# Option C: For private repos, authenticate with gh first
# !gh auth login
# !gh repo clone your-username/car-census
# import os
# os.chdir('car-census')

In [ ]:
# Option B: Upload via file picker
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# Option C: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/car-census /content/

## 2. Install Dependencies

In [ ]:
%cd /content/Car-Census

!pip install -e ".[tracking]"
!pip uninstall -y onnxruntime
!pip install -U onnxruntime-gpu

## 3. Verify GPU, ONNX Runtime & FFmpeg

In [ ]:
import torch
import onnxruntime as ort

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
print(f"ONNX Runtime providers: {ort.get_available_providers()}")
!ffmpeg -hide_banner -encoders | grep nvenc || true

## 3. Upload Data

Upload your test video and model weights:

In [ ]:
# Option A: Upload via file picker
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# Option B: Copy from Google Drive (if mounted)
# !cp /content/drive/MyDrive/test4K.MP4 /content/car-census/input_data/

## 4. Set Environment Variables

In [ ]:
traffic_eye_api_key = userdata.get('TRAFFICEYE_API_KEY')
if not traffic_eye_api_key:
    raise RuntimeError('Add TRAFFICEYE_API_KEY to Colab Secrets before running classification.')

os.environ['TRAFFICEYE_API_KEY'] = traffic_eye_api_key
print('TRAFFICEYE_API_KEY loaded from Colab Secrets.')

## 5. Run Car-Census

### Option A: Run Full Pipeline (detect, track, classify, render)

In [ ]:
!Car-Census run input_data/CarsHighwayTraffic_1440_10s.mp4 --accelerator colab-t4

### Option B: Run Without Classification (skip API calls)

In [ ]:
!Car-Census run input_data/CarsHighwayTraffic_1440_10s.mp4 --skip-classify --accelerator colab-t4

### Option C: Run with Camera ID

In [ ]:
!Car-Census run input_data/test4K.MP4 --camera-id my-camera --accelerator colab-t4

### Option D: ROI Edit (define polygon zone)

In [ ]:
!Car-Census roi edit input_data/test4K.MP4 --camera-id my-camera --device cuda

## 6. Download Results

In [ ]:
# Download output folder
from google.colab import files

!zip -r output.zip output/
files.download('output.zip')

In [ ]:
# Or download specific file
# files.download('/content/car-census/outputs/<run-id>/annotated.mp4')